In [ ]:
# ============================================================
# DEEP_DIVE_4CONN_V2: ГЛУБОКИЙ РАСЧЁТ 4-СВЯЗНОСТИ (2.5 МЛН ШАГОВ)
# ============================================================
# REM: Этот ноутбук считает 4 "вкусные" точки для ЖЁСТКОЙ 4-связности.
# REM: Взаимодействие — только 4 ближайших соседа (нет дальнего поля).
# REM: Энергия системы: S = -λ Σ N² + σ · Perimeter (см. функцию action ниже).
# REM: Внутри цикла стоит for _ in range(5), поэтому 500 000 × 5 = 2 500 000 реальных шагов.

# REM: Импорт numpy для работы с массивами и случайными числами.
import numpy as np
# REM: Импорт label из scipy для поиска связных кластеров (4-связность).
from scipy.ndimage import label
# REM: Импорт matplotlib для отрисовки картинок.
import matplotlib
# REM: Включаем фоновый режим 'Agg', чтобы графики сохранялись в файлы, а не открывались в окнах.
matplotlib.use('Agg')
import matplotlib.pyplot as plt
# REM: Импорт csv для записи таблиц с результатами.
import csv
# REM: Импорт time для замера времени выполнения расчётов.
import time
# REM: Импорт os для работы с файловой системой (создание папок, проверка путей).
import os
# REM: Импорт display и Image для вывода картинок прямо в Jupyter.
from IPython.display import display, Image as IPImage

# ------------------------------------------------------------
# REM: ПАПКИ ДЛЯ РЕЗУЛЬТАТОВ И ЧЕКПОИНТОВ
# ------------------------------------------------------------
# REM: output_dir — корневая папка, куда пишутся все результаты.
output_dir = "DEEP_DIVE_4CONN_V2"
# REM: csv_dir — подпапка для таблиц с числами (CSV).
csv_dir = os.path.join(output_dir, "csv")
# REM: png_dir — подпапка для картинок решётки (PNG).
png_dir = os.path.join(output_dir, "png")
# REM: ckpt_dir — подпапка для чекпоинтов (сохранений сетки для восстановления после сбоя).
ckpt_dir = os.path.join(output_dir, "checkpoints")
# REM: Создаём все папки. exist_ok=True предотвращает ошибку, если папка уже существует.
os.makedirs(output_dir, exist_ok=True)
os.makedirs(csv_dir, exist_ok=True)
os.makedirs(png_dir, exist_ok=True)
os.makedirs(ckpt_dir, exist_ok=True)

# ------------------------------------------------------------
# REM: ПАРАМЕТРЫ МОДЕЛИ
# ------------------------------------------------------------
# REM: L — размер решётки (100×100 = 10 000 узлов/плексонов).
L = 100
# REM: STEPS — число итераций ВНЕШНЕГО цикла.
# REM: ВАЖНО: внутри цикла стоит for _ in range(5), поэтому реальное число шагов = 500 000 × 5 = 2 500 000.
STEPS = 500_000
# REM: beta — обратная температура (1/kT). Отвечает за тепловой шум.
# REM: Шаг Метрополиса принимается с вероятностью exp(-beta·ΔS), если энергия выросла.
beta = 1.0
# REM: lam (лямбда) — СИЛА АГРЕГАЦИИ в 4-связности.
# REM: В энергии S = -λ Σ N² знак минус означает: чем больше кластер (N), тем выгоднее (ниже энергия).
# REM: λ = 3.0 — режим сильного притяжения, доминирующего над тепловым шумом β = 1.0.
lam = 3.0

# REM: POINTS — список "вкусных" точек (плотность, натяжение σ), выбранных по скринингу.
# REM: (0.10, 2.0) — 10% плотности, σ=2.0 — «дно пилы» (минимум периметра).
# REM: (0.10, 2.5) — 10% плотности, σ=2.5 — «скачок пилы» (резкий рост периметра).
# REM: (0.50, 1.5) — 50% плотности, σ=1.5 — начало образования геля.
# REM: (0.50, 2.0) — 50% плотности, σ=2.0 — стабильный гель (плато).
POINTS = [
    (0.10, 2.0),
    (0.10, 2.5),
    (0.50, 1.5),
    (0.50, 2.0)
]

# REM: csv_file — путь к таблице результатов.
csv_file = os.path.join(csv_dir, "deep_dive_4conn_v2.csv")

# REM: Если файла ещё нет — создаём его и записываем заголовки колонок.
if not os.path.exists(csv_file):
    with open(csv_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['p_init', 'lam', 'sigma', 'step', 'density', 'P_inf', 'perimeter', 'max_size'])

# ------------------------------------------------------------
# REM: ФИЗИЧЕСКИЕ ФУНКЦИИ
# ------------------------------------------------------------

# REM: init_lattice — создаёт случайную решётку: с вероятностью p узел занят (1 = плексон), иначе пуст (0).
def init_lattice(L, p):
    return (np.random.random((L, L)) < p).astype(int)

# REM: find_clusters_periodic — ищет связные кластеры (4-связность) с учётом периодических границ (тор).
# REM: np.pad с mode='wrap' добавляет рамку-копию, чтобы кластер "видел" себя через край.
# REM: structure=[[0,1,0],[1,1,1],[0,1,0]] задаёт связность по 4 граням (без диагоналей).
def find_clusters_periodic(grid):
    padded = np.pad(grid, pad_width=1, mode='wrap')
    labeled, n = label(padded, structure=[[0,1,0],[1,1,1],[0,1,0]])
    labeled = labeled[1:-1, 1:-1]   # REM: убираем добавленную рамку.
    sizes = np.bincount(labeled.ravel())  # REM: размеры кластеров.
    sizes[0] = 0                     # REM: метка 0 = пустота, её размер не считаем.
    return labeled, sizes

# REM: calculate_perimeter_periodic — считает ПЕРИМЕТР (суммарную длину границы "материя/пустота").
# REM: np.roll сдвигает решётку на 1; где занятая клетка (1) касается пустой (0) — там граница.
# REM: Чем ниже периметр, тем компактнее структура (меньше рваных границ).
def calculate_perimeter_periodic(grid):
    right = np.roll(grid, 1, axis=1)
    left = np.roll(grid, -1, axis=1)
    up = np.roll(grid, 1, axis=0)
    down = np.roll(grid, -1, axis=0)
    p = 0
    p += np.sum((grid == 1) & (right == 0))
    p += np.sum((grid == 1) & (left == 0))
    p += np.sum((grid == 1) & (up == 0))
    p += np.sum((grid == 1) & (down == 0))
    return int(p)

# REM: action — ПОЛНАЯ энергия системы S = -λ Σ N² + σ · Perimeter.
# REM: S_bulk = -λ Σ N² — агрегация: слипание выгодно (минус).
# REM: S_surface = σ · Perimeter — поверхностное натяжение: штраф за рваные границы (плюс).
def action(grid, lam, sigma):
    _, sizes = find_clusters_periodic(grid)
    S_bulk = -lam * np.sum(sizes**2)
    S_surface = sigma * calculate_perimeter_periodic(grid)
    return S_bulk + S_surface

# REM: metropolis_step_swap — ОДИН шаг Метрополиса (обмен занятого и пустого узла).
# REM: Обмен сохраняет плотность: общее число плексонов не меняется.
def metropolis_step_swap(grid, lam, sigma, beta=1.0):
    L = grid.shape[0]
    # REM: случайно ищем ЗАНЯТЫЙ узел (1).
    while True:
        i1, j1 = np.random.randint(0, L, 2)
        if grid[i1, j1] == 1: break
    # REM: случайно ищем ПУСТОЙ узел (0).
    while True:
        i2, j2 = np.random.randint(0, L, 2)
        if grid[i2, j2] == 0: break
    # REM: энергия ДО обмена.
    S_old = action(grid, lam, sigma)
    # REM: меняем узлы местами.
    grid[i1, j1] = 0
    grid[i2, j2] = 1
    # REM: энергия ПОСЛЕ обмена.
    S_new = action(grid, lam, sigma)
    delta_S = S_new - S_old
    # REM: критерий Метрополиса: принимаем, если энергия упала, или с вероятностью Больцмана exp(-βΔS).
    if delta_S <= 0 or np.random.random() < np.exp(-beta * delta_S):
        return True
    else:
        # REM: если не приняли — возвращаем узлы на место.
        grid[i1, j1] = 1
        grid[i2, j2] = 0
        return False

# REM: measure — измеряет макроскопические характеристики системы.
# REM: P_inf — доля узлов в САМОМ БОЛЬШОМ кластере (0 = всё рассеяно, 1 = монолит).
# REM: density — реальная измеренная плотность (должна совпадать с заданной).
# REM: perimeter — периметр (граница материя/пустота).
# REM: max_size — абсолютный размер наибольшего кластера.
def measure(grid):
    labeled, sizes = find_clusters_periodic(grid)
    sizes_nonzero = sizes[sizes > 0]
    if len(sizes_nonzero) == 0: return None
    P_inf = float(sizes_nonzero.max() / np.sum(sizes_nonzero))
    density = float(np.mean(grid))
    perimeter = calculate_perimeter_periodic(grid)
    max_size = int(sizes_nonzero.max())
    return {'density': density, 'P_inf': P_inf, 'perimeter': perimeter, 'max_size': max_size}

# ------------------------------------------------------------
# REM: ЗАПУСК С ПОДДЕРЖКОЙ ВОССТАНОВЛЕНИЯ (ЧЕКПОИНТЫ)
# ------------------------------------------------------------
# REM: Перебираем все 4 точки.
for p, sigma in POINTS:
    # REM: пути к файлам чекпоинтов (сохранённая сетка и номер шага).
    ckpt_grid_file = os.path.join(ckpt_dir, f"grid_p{p}_s{sigma}.npy")
    ckpt_step_file = os.path.join(ckpt_dir, f"step_p{p}_s{sigma}.npy")

    start_step = 0
    grid = None

    # REM: если есть сохранение — восстанавливаемся с него.
    if os.path.exists(ckpt_grid_file) and os.path.exists(ckpt_step_file):
        grid = np.load(ckpt_grid_file)
        start_step = int(np.load(ckpt_step_file)[0])
        print(f"\n[RESTART] Плотность {p*100}%, Сигма {sigma}. Восстанавливаем с шага {start_step*5}...", flush=True)
    elif os.path.exists(csv_file):
        # REM: если сохранения нет — проверяем, не досчитана ли точка (по финальной строке в CSV).
        is_done = False
        with open(csv_file, 'r') as f:
            reader = csv.reader(f)
            next(reader, None)
            for row in reader:
                if len(row) >= 4 and abs(float(row[0]) - p) < 1e-5 and abs(float(row[2]) - sigma) < 1e-5:
                    if int(row[3]) == STEPS * 5: is_done = True
        if is_done:
            print(f"[SKIP] Плотность {p*100}%, Сигма {sigma} уже полностью посчитана.", flush=True)
            continue
        else:
            grid = init_lattice(L, p)
    else:
        grid = init_lattice(L, p)

    # REM: печатаем заголовок таблицы для текущей точки.
    print("\n" + "="*65, flush=True)
    print(f"СЧИТАЕМ (4-связность): Плотность = {p*100}%, Сигма = {sigma}, Лямбда = {lam}", flush=True)
    print("="*65, flush=True)
    print(f"{'Шаг':>8} | {'density':>8} | {'P_inf':>8} | {'Perimeter':>10} | {'Max_Size':>8} | {'Time(s)':>7}", flush=True)
    print("-" * 65, flush=True)

    # REM: засекаем время начала расчёта точки.
    t0 = time.time()

    # REM: основной цикл (начинается с start_step при восстановлении).
    for step in range(start_step, STEPS):
        # REM: внутренний цикл делает 5 шагов Метрополиса за 1 итерацию.
        for _ in range(5):
            metropolis_step_swap(grid, lam, sigma, beta)

        # REM: реальное число сделанных шагов = step * 5.
        real_step = step * 5

        # REM: каждые 250 000 реальных шагов — замер, запись в CSV, чекпоинт и картинка.
        if real_step % 250_000 == 0 and real_step > 0:
            m = measure(grid)
            elapsed = time.time() - t0

            with open(csv_file, 'a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([p, lam, sigma, real_step, m['density'], m['P_inf'], m['perimeter'], m['max_size']])

            print(f"{real_step:>8} | {m['density']:>8.3f} | {m['P_inf']:>8.3f} | {m['perimeter']:>10.1f} | {m['max_size']:>8d} | {elapsed:>7.1f}", flush=True)

            # REM: сохраняем сетку и номер шага (чекпоинт).
            np.save(ckpt_grid_file, grid)
            np.save(ckpt_step_file, np.array([step]))

            # REM: рисуем и сохраняем картинку текущего состояния (чёрный = плексон, белый = пустота).
            fig, ax = plt.subplots(figsize=(4, 4))
            ax.imshow(grid, cmap='binary', interpolation='nearest')
            ax.set_title(f'4-conn D={p*100}%, σ={sigma}, Step={real_step}')
            img_path = os.path.join(png_dir, f"C_D{p}_S{sigma}_step{real_step}.png")
            plt.savefig(img_path, dpi=100, bbox_inches='tight')
            plt.close(fig)
            display(IPImage(filename=img_path))

    # REM: после цикла — финальный замер и запись финальной строки.
    m = measure(grid)
    with open(csv_file, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([p, lam, sigma, STEPS*5, m['density'], m['P_inf'], m['perimeter'], m['max_size']])

    # REM: удаляем чекпоинты (точка полностью досчитана).
    if os.path.exists(ckpt_grid_file): os.remove(ckpt_grid_file)
    if os.path.exists(ckpt_step_file): os.remove(ckpt_step_file)

    print(f"ТОЧКА ЗАВЕРШЕНА.", flush=True)

print("\nВСЕ 4 ТОЧКИ 4-СВЯЗНОСТИ ГОТОВЫ! Данные в DEEP_DIVE_4CONN_V2.", flush=True)
